# NYC Motor Vehicle Collisions: EDA and Feature Engineering

This notebook performs exploratory data analysis and feature engineering for the NYC Motor Vehicle Collisions final project. The goal is to prepare a cleaned and engineered dataset that can later be used for supervised machine learning classification.

The target variable is whether a crash resulted in at least one injury.

## 1. Import Libraries

In [ ]:
import sys
sys.path.append("../src")
from features import run_engineering
from preprocessing import run_prep
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

## 2. Load Dataset

The project dataset is already merged into one file called `merged_sample.csv`.

In [ ]:
df = pd.read_csv("../data/raw/merged_sample.csv")

print("Dataset Shape:", df.shape)
display(df.head())

## 3. Standardize Column Names

Column names are standardized to make them easier to use in Python. Spaces are replaced with underscores and all names are converted to lowercase.

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(df.columns.tolist())

## 4. Basic Dataset Inspection

In [ ]:
print("Dataset Info:")
df.info()

print("\nMissing Values:")
display(df.isnull().sum().sort_values(ascending=False).head(25))

print("\nDuplicate Rows:")
print(df.duplicated().sum())

## 5. Create Target Variable

The target variable is called `injury_occurred`. It equals 1 if at least one person was injured in the crash, and 0 otherwise.

In [ ]:
injury_col = "number_of_persons_injured"

if injury_col not in df.columns:
    raise ValueError(f"Expected column '{injury_col}' was not found. Check the column names above.")

df["injury_occurred"] = (df[injury_col] > 0).astype(int)

print("Target Distribution:")
display(df["injury_occurred"].value_counts())

print("Target Percentage:")
display(df["injury_occurred"].value_counts(normalize=True) * 100)

## 6. EDA: Injury Distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="injury_occurred")
plt.title("Injury vs No Injury")
plt.xlabel("Injury Occurred")
plt.ylabel("Count")
plt.show()

## 7. Feature Engineering: Date and Time Features

New time-based features are created from `crash_date` and `crash_time`, including crash hour, day of week, and month.

In [ ]:
df["crash_date"] = pd.to_datetime(df["crash_date"], errors="coerce")

df["crash_time"] = pd.to_datetime(
    df["crash_time"],
    format="%H:%M",
    errors="coerce"
)

df["hour"] = df["crash_time"].dt.hour
df["day_name"] = df["crash_date"].dt.day_name()
df["month"] = df["crash_date"].dt.month

display(df[["crash_date", "crash_time", "hour", "day_name", "month"]].head())

## 8. EDA: Crash Frequency by Hour

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["hour"].dropna(), bins=24)
plt.title("Crash Frequency by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Crash Count")
plt.show()

## 9. EDA: Crashes by Day of Week

In [ ]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="day_name", order=day_order)
plt.title("Crashes by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Crash Count")
plt.xticks(rotation=45)
plt.show()

## 10. EDA: Borough Analysis

In [ ]:
if "borough" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.countplot(
        data=df,
        y="borough",
        order=df["borough"].value_counts().index
    )
    plt.title("Crashes by Borough")
    plt.xlabel("Crash Count")
    plt.ylabel("Borough")
    plt.show()
else:
    print("No borough column found. Skipping borough count plot.")

## 11. EDA: Injury Rate by Borough

In [ ]:
if "borough" in df.columns:
    borough_rates = (
        df.groupby("borough")["injury_occurred"]
        .mean()
        .sort_values(ascending=False)
    )

    display(borough_rates)

    plt.figure(figsize=(8, 4))
    borough_rates.plot(kind="bar")
    plt.title("Injury Rate by Borough")
    plt.xlabel("Borough")
    plt.ylabel("Injury Rate")
    plt.xticks(rotation=45)
    plt.show()
else:
    print("No borough column found. Skipping injury rate by borough.")

## 12. Feature Engineering: Rush Hour and Weekend Features

Two binary features are created:

- `rush_hour`: 1 if the crash happened during common morning or evening commute hours
- `is_weekend`: 1 if the crash happened on Saturday or Sunday

In [ ]:
# Calling consolidated feature engineering from src/features.py
df = run_engineering(df)
display(df[["hour", "is_rush_hour", "day_name", "is_weekend"]].head())


## 13. EDA: Vehicle Type and Person Type

These checks are included only if the merged dataset contains vehicle or person-related columns.

In [ ]:
if "vehicle_type_code" in df.columns:
    top_vehicles = (
        df["vehicle_type_code"]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", np.nan)
        .dropna()
        .value_counts()
        .head(10)
    )

    if len(top_vehicles) > 0:
        display(top_vehicles)

        plt.figure(figsize=(8, 5))
        top_vehicles.plot(kind="bar")
        plt.title("Top 10 Vehicle Types")
        plt.xlabel("Vehicle Type")
        plt.ylabel("Count")
        plt.xticks(rotation=45)
        plt.show()
    else:
        print("vehicle_type_code column exists, but no usable values were found.")
else:
    print("No vehicle_type_code column found. Skipping vehicle type plot.")


if "person_type" in df.columns:
    person_type_counts = (
        df["person_type"]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", np.nan)
        .dropna()
        .value_counts()
        .head(10)
    )

    if len(person_type_counts) > 0:
        display(person_type_counts)

        plt.figure(figsize=(8, 5))
        person_type_counts.plot(kind="bar")
        plt.title("Person Type Distribution")
        plt.xlabel("Person Type")
        plt.ylabel("Count")
        plt.xticks(rotation=45)
        plt.show()
    else:
        print("person_type column exists, but no usable values were found.")
else:
    print("No person_type column found. Skipping person type plot.")

## 14. Missing Value Handling

Missing categorical values are filled with `Unknown`. Missing numeric values are filled with 0 for this initial feature engineering stage.

In [ ]:
categorical_cols = df.select_dtypes(include="object").columns
numeric_cols = df.select_dtypes(include=np.number).columns

df[categorical_cols] = df[categorical_cols].fillna("Unknown")
df[numeric_cols] = df[numeric_cols].fillna(0)

print("Remaining missing values:", df.isnull().sum().sum())

## 15. Final Engineered Feature Preview

In [ ]:
selected_features = [
    "injury_occurred",
    "hour",
    "day_name",
    "month",
    "is_rush_hour",
    "is_weekend",
    "borough",
    "vehicle_type_code"
]

existing_features = [col for col in selected_features if col in df.columns]

display(df[existing_features].head())

print("Final Dataset Shape:", df.shape)

## 16. Save Engineered Dataset

The engineered dataset is saved as `engineered_collisions.csv` so the next step of the project can use it for model training.

In [ ]:
df.to_csv("engineered_collisions.csv", index=False)

print("Saved engineered_collisions.csv")

## 17. Summary

In this notebook, the merged NYC Motor Vehicle Collisions dataset was loaded, inspected, and prepared for modeling. A binary injury target was created, basic crash patterns were explored, and new time-based features were engineered. The final engineered dataset was saved for the model training stage of the project.